# MedMatch Model A — MedGemma 1.5 4B LoRA/QLoRA fine-tune (Colab)

GPU-adaptive SFT for clinical note extraction. **Model A only** (not Model B / trial matching).

**Before running:** `Runtime → Change runtime type → GPU` (prefer **G4**, then **A100**, then L4/T4).
Add your Hugging Face token to **Colab Secrets** as `HF_TOKEN` (toggle notebook access on).
You must have accepted gated access to `google/medgemma-1.5-4b-it` on the HF account that owns the token.

This notebook mirrors `colab_bootstrap.py`; either runs the same pipeline.

In [ ]:
# 1. GPU + disk inventory
!nvidia-smi || echo 'No GPU — set Runtime > Change runtime type > GPU'
!df -h /content | tail -1
import torch
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print({'name': p.name, 'vram_gb': round(p.total_memory/1024**3, 2),
           'cuda': torch.version.cuda, 'cc': f'{p.major}.{p.minor}',
           'bf16': torch.cuda.is_bf16_supported()})

In [ ]:
# 2. Install pinned dependencies
!pip -q install -U 'transformers>=4.44' 'peft>=0.12' 'trl>=0.9' 'bitsandbytes>=0.43' \
    'accelerate>=0.33' 'datasets>=2.20' huggingface_hub pyyaml

In [ ]:
# 3. Check out the project (pinned branch)
REPO = 'https://github.com/RayyanSarfraz/Hackachusetts----Rayyan-Muhammad-Camden.git'
REF = 'feature/model-a-medgemma-finetune'
!rm -rf /content/medmatch && git clone --depth 1 -b $REF $REPO /content/medmatch
%cd /content/medmatch/ml/model_a
!git rev-parse --short HEAD

In [ ]:
# 4. HF auth from Colab Secrets (token is never printed)
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
from huggingface_hub import HfApi
info = HfApi(token=os.environ['HF_TOKEN']).model_info('google/medgemma-1.5-4b-it')
print('HF access OK, model sha', info.sha[:8])

In [ ]:
# 5. Mount Drive for persistent artifacts
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/medgemma-model-a

In [ ]:
# 6. Prepare data (deterministic ingest, filter, dedup, case-level split, manifest)
!python prepare_data.py --out /content/data --primary-n 1000 --secondary-n 150
!cat /content/data/manifest.json

In [ ]:
# 7. Print the EFFECTIVE config (GPU profile merged over defaults)
!python train.py --print-effective-only

In [ ]:
# 8. Smoke test: single forward/backward, logs peak VRAM + tokens/sec
!python train.py --data /content/data --out /content/outputs --smoke

In [ ]:
# 9. Full LoRA training (adapter persisted to Drive)
!python train.py --data /content/data --out /content/outputs \
    --drive /content/drive/MyDrive/medgemma-model-a

In [ ]:
# 10. Evaluate: unchanged base first, then base+adapter, same greedy decoding
!python evaluate.py --data /content/data --adapter /content/outputs/adapter \
    --out /content/outputs/metrics --limit 40
!cp -r /content/outputs/metrics /content/drive/MyDrive/medgemma-model-a/
!cat /content/outputs/metrics/metrics.json

In [ ]:
# 11. Reload clean base + adapter and infer on a held-out example (valid JSON)
import json
rows = [json.loads(l) for l in open('/content/data/test.jsonl')]
sample = next(r for r in rows if r['task'] == 'EXTRACT_SUMMARY')
open('/content/sample.txt', 'w').write(sample['source_text'])
!python infer.py --adapter /content/outputs/adapter --task both --text-file /content/sample.txt

## 12. Stop the VM
Artifacts (adapter, metrics, logs, manifest) are in Drive at `MyDrive/medgemma-model-a`.
Stop the runtime from the CLI (`colab stop -s medgemma-model-a`) or `Runtime → Disconnect and delete runtime` to stop consuming compute units.